In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import altair as alt
from datetime import datetime

In [ ]:
sge_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251202_SGEsubset.xlsx')
vampseq_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251204_VAMPseqsubset_wDups.xlsx')


dfs = {'SGE': sge_ppj_subset,
       'VAMPseq': vampseq_ppj_subset
      }

alt.data_transformers.disable_max_rows() #gets rid of max data length problem

In [ ]:
def process_dfs(dfs):

    keys = list(dfs.keys())
    cleaned_dfs = {}
    for key in keys:
        df = dfs[key]
        df = df.rename(columns = {'auth_reported_func_class': 'functional_consequence',
                                   'clinvar_sig': 'Germline classification'}
                      )

        df = df.dropna(subset = ['functional_consequence'])
        df = df.dropna(subset = ['Germline classification'])

        df = df.loc[(df['functional_consequence'] != 'indeterminate') & (~df['Germline classification'].isin(['Uncertain significance', 'Conflicting classifications of pathogenicity']))]

        cleaned_dfs[key] = df

    return cleaned_dfs

In [ ]:
def wilson_interval(successes, total, confidence=0.95):
    """
    Calculate Wilson score confidence interval for a proportion
    """
    if total == 0:
        return 0, 0, 0
    
    p = successes / total
    z = stats.norm.ppf((1 + confidence) / 2)
    
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    margin = z * np.sqrt((p * (1 - p) / total + z**2 / (4 * total**2))) / denominator
    
    lower = center - margin
    upper = center + margin
    
    return p, lower, upper

In [ ]:
def clinvar_performance(df, assay_name):

    df = df.loc[~df['functional_consequence'].isin(['indeterminate'])]

    df = df.loc[~df['Germline classification'].isin(['Uncertain significance', 'Conflicting classifications of pathogenicity', 'no classification for the single variant', 'not provided'])]

    df = df.dropna(subset = ['Germline classification'])
    df['test'] = 0
    df.loc[(df['Germline classification'] == 'Benign') | (df['Germline classification'] == 'Likely benign') | (df['Germline classification'] == 'Benign/Likely benign') | (df['Germline classification'] == 'Benign; association'), 'simple_classification'] = 'Benign'
    df.loc[(df['Germline classification'] == 'Pathogenic') | (df['Germline classification'] == 'Likely pathogenic') | (df['Germline classification'] == 'Pathogenic/Likely pathogenic'), 'simple_classification'] = 'Pathogenic'
    
    benign_df = df.loc[df['simple_classification'].isin(['Benign'])] #Number of variants truly benign from ClinVar (negative)
    path_df = df.loc[df['simple_classification'].isin(['Pathogenic'])] #Number of variants truly pathogenic from ClinVar (positive)

    assay_benign_df = df.loc[df['functional_consequence'].isin(['functionally_normal'])] #Number of variants called benign by assay
    assay_path_df =  df.loc[df['functional_consequence'].isin(['functionally_abnormal'])] #Number of variants called abnormal by assay

    benign_df.loc[benign_df['functional_consequence'] == 'functionally_normal', 'test'] = 1
    path_df.loc[path_df['functional_consequence'] == 'functionally_abnormal', 'test'] = 1

    assay_benign_df.loc[assay_benign_df['simple_classification'] == 'Benign', 'test'] = 1 
    assay_path_df.loc[assay_path_df['simple_classification'] == 'Pathogenic', 'test'] = 1

    perc_benign = (benign_df['test'].sum() / len(benign_df)) * 100 #Calculates percent of true negative results called benign by assay
    perc_path = (path_df['test'].sum() / len(path_df)) * 100 #Calculates percent of true positive results called abnormal by assay

    '''
    print('% Benign Correct: ', str(perc_benign), '\n',
        '% Path. Correct: ', str(perc_path), '\n')

    df = pd.DataFrame({'var_type': ['benign', 'pathogenic'],
                        'correct': [perc_benign, perc_path]
                      }
                     )

    plot = alt.Chart(df).mark_bar().encode(
        x = alt.X('var_type:O',
                  title = 'Germline Classification'
                 ),
        y = alt.Y('correct:Q', 
                  title = '% Agreement',
                  scale = alt.Scale(domain = [0, 100])
                 )
    )
    plot.display()
    '''
    
    # Calculate precision and recall with confidence intervals
    npv_successes = assay_benign_df['test'].sum()
    npv_total = len(assay_benign_df)
    raw_npv, npv_lower, npv_upper = wilson_interval(npv_successes, npv_total)
    
    ppv_successes = assay_path_df['test'].sum()
    ppv_total = len(assay_path_df)

    raw_ppv, ppv_lower, ppv_upper = wilson_interval(ppv_successes, ppv_total)
    
    perc_benign_agree = raw_npv * 100
    perc_path_agree = raw_ppv * 100

    recall_successes = path_df['test'].sum()
    recall_total = len(path_df)
    recall, recall_lower, recall_upper = wilson_interval(recall_successes, recall_total)

    print(f'Precision/PPV: {raw_ppv:.3f} (95% CI: {ppv_lower:.3f}-{ppv_upper:.3f})')
    print(f'Recall/Sensitivity: {recall:.3f} (95% CI: {recall_lower:.3f}-{recall_upper:.3f})')
    print(f'NPV: {raw_npv:.3f} (95% CI: {npv_lower:.3f}-{npv_upper:.3f})')
    
    
    # Create dataframe with error bars
    agree_df = pd.DataFrame({
        'assay_type': ['Recall', 'Precision'],
        'correct': [recall, raw_ppv],
        'lower_ci': [recall_lower, ppv_lower],
        'upper_ci': [recall_upper, ppv_upper],
        'error_lower': [recall - recall_lower, raw_ppv - ppv_lower],
        'error_upper': [recall_upper - recall, ppv_upper - raw_ppv]
    })
    
    # Bar chart
    bars = alt.Chart(agree_df).mark_bar(color = 'gray').encode(
        y = alt.Y('assay_type:N',
                  title = '',
                  sort = ['Precision', 'Recall'],
                  axis = alt.Axis(ticks = False,
                                  labelFontSize = 16,
                                  labelFontWeight = 'bold'
                                 )
                 ),
        x = alt.X('correct:Q',
                  title = '', 
                  axis = alt.Axis(titleFontSize = 20, 
                                  labelFontSize = 16
                                 ),
                  scale = alt.Scale(domain = [0, 1])
                 ),
        color = alt.Color('assay_type:N',
                          scale = alt.Scale(
                           domain = ['Precision', 'Recall'],
                           range = ['#888888', '#888888' ]
                          ),
                          legend = None
                         )
    )
    
    # Error bars
    error_bars = alt.Chart(agree_df).mark_errorbar(thickness = 2).encode(
        y = alt.Y('assay_type:O', sort = ['Precision', 'Recall']),
        x = alt.X('lower_ci:Q'),
        x2 = alt.X2('upper_ci:Q'),
        color = alt.value('black')
    )
    
    # Combine
    plot = (bars + error_bars).properties(
        width = 300, 
        height = 50
    ).configure_axis(
        grid = False
    ).configure_view(
        stroke = None
    )
    
    plot.display()
    date = datetime.today().strftime('%Y%m%d')
    save_string = f'/Users/ivan/Desktop/pillar_project_figs/{date}_{assay_name}_PRvsClinVar_wErrorBar_grey.svg'

    #plot.save(save_string)

In [ ]:
def main():
    processed_dfs = process_dfs(dfs)

    keys = ['SGE', 'VAMPseq']
    
    for key in keys:
        df = processed_dfs[key]

        clinvar_performance(df, key)

In [ ]:
main()